# Section 10: Ship Dock

Goal: Dock runs as a web service other systems can call. It never spends more than its budget, answers in time
or hands over to a person, never sends the same email twice, and a bad release is undone in one line.

## Step 1. A first endpoint

In [1]:
import json, time
from fastapi import FastAPI, Header, HTTPException
from fastapi.testclient import TestClient
from dock import app as dock

api = FastAPI(title="Dock")

@api.post("/triage")
def triage(mail: dict):
    t = dock.triage(mail)
    return t.model_dump()

client = TestClient(api)
inbox = json.loads(open("dock/inbox.json", encoding="utf-8").read())
start = time.time()
r = client.post("/triage", json=inbox[1])
print(r.status_code, r.json(), f"{time.time() - start:.1f} s")

200 {'intent': 'customs', 'shipment_id': 'HF-26038', 'language': 'de'} 0.9 s


## Step 2. A token budget

In [2]:
class Budget:
    def __init__(self, tokens_per_day, reserve):
        self.limit, self.reserve, self.used = tokens_per_day, reserve, 0

    def check(self):          # before the call: is there room for one more?
        if self.used + self.reserve > self.limit:
            raise HTTPException(429, f"daily budget of {self.limit} tokens reached")

    def spend(self, tokens):  # after the call: count what it really used
        self.used += tokens

budget = Budget(tokens_per_day=300, reserve=150)


@api.post("/triage/v2")
def triage_v2(mail: dict):
    budget.check()
    r = dock.client.chat.parse(model="mistral-small-latest", response_format=dock.Triage, messages=[
        {"role": "system", "content": "You triage emails for Harbor Freight Co."},
        {"role": "user", "content": mail["subject"] + "\n\n" + mail["body"]}])
    budget.spend(r.usage.total_tokens)
    return {**r.choices[0].message.parsed.model_dump(), "tokens": r.usage.total_tokens}

for mail in inbox[:5]:
    r = client.post("/triage/v2", json=mail)
    print(mail["id"], r.status_code, r.json().get("tokens", r.json().get("detail")), "| used", budget.used)

M01 200 85 | used 85


M02 200 105 | used 190
M03 429 daily budget of 300 tokens reached | used 190
M04 429 daily budget of 300 tokens reached | used 190
M05 429 daily budget of 300 tokens reached | used 190


## Step 3. Answer in time, or hand over

In [3]:
import asyncio

FALLBACK = {"status": "queued", "message": "A person from operations will reply."}

@api.post("/answer")
async def answer(body: dict, seconds: float = 8.0):
    try:
        a, sources = await asyncio.wait_for(asyncio.to_thread(dock.answer, body["question"]), timeout=seconds)
    except asyncio.TimeoutError:
        return FALLBACK
    return {"status": "answered", "answer": a.answer, "sources": a.sources}

q = {"question": "How many free days do we get at Los Angeles before demurrage starts?"}
for seconds in [8.0, 0.2]:
    start = time.time()
    r = client.post(f"/answer?seconds={seconds}", json=q)
    print(f"limit {seconds:>3} s ->", r.json(), f"({time.time() - start:.1f} s)")

limit 8.0 s -> {'status': 'answered', 'answer': 'Four days', 'sources': ['HB-07']} (3.8 s)


limit 0.2 s -> {'status': 'queued', 'message': 'A person from operations will reply.'} (1.0 s)


## Step 4. Never send the same email twice

In [4]:
SEEN = {}

@api.post("/notify")
def notify(body: dict, idempotency_key: str = Header(...)):
    if idempotency_key in SEEN:
        return {**SEEN[idempotency_key], "repeated": True}
    dock.send_email(body["to"], body["subject"], body["body"])
    SEEN[idempotency_key] = {"status": "sent", "to": body["to"]}
    return SEEN[idempotency_key]

dock.OUTBOX.clear()
msg = {"to": "import@nordlicht.example.com", "subject": "HF-26038 update", "body": "Docs are with the broker."}
for attempt in range(3):          # the caller retries after a timeout
    r = client.post("/notify", json=msg, headers={"Idempotency-Key": "HF-26038-update-1"})
    print("attempt", attempt + 1, r.json())
print("emails actually sent:", len(dock.OUTBOX))

attempt 1 {'status': 'sent', 'to': 'import@nordlicht.example.com'}
attempt 2 {'status': 'sent', 'to': 'import@nordlicht.example.com', 'repeated': True}
attempt 3 {'status': 'sent', 'to': 'import@nordlicht.example.com', 'repeated': True}
emails actually sent: 1


## Step 5. Versions and a one-line rollback

In [5]:
mistral = dock.client
agent = mistral.beta.agents.create(
    model="mistral-small-latest", name="dock-reply",
    instructions="You are Dock. Reply in one sentence. Never promise a delivery date.")
v2 = mistral.beta.agents.update(
    agent_id=agent.id,
    instructions="You are Dock. Reply in one sentence. Reassure the customer with a firm delivery date.")
mistral.beta.agents.create_version_alias(agent_id=agent.id, alias="prod", version=v2.version)
print("versions:", agent.version, "and", v2.version, "· prod ->", v2.version)


@api.post("/reply")
def reply(body: dict):
    r = mistral.beta.conversations.start(agent_id=agent.id, agent_version="prod",
                                         inputs=body["text"], store=False)
    text = str(r.outputs[-1].content)
    return {"reply": text, "guarantees_date": dock.guarantees_date(text)}

ask = {"text": "Will HF-26012 arrive in Felixstowe by Friday?"}
print(client.post("/reply", json=ask).json())

versions: 0 and 1 · prod -> 1


{'reply': 'HF-26012 is scheduled to arrive in Felixstowe by Friday, October 30, 2026.', 'guarantees_date': True}


In [6]:
mistral.beta.agents.create_version_alias(agent_id=agent.id, alias="prod", version=agent.version)
print("prod ->", agent.version)
print(client.post("/reply", json=ask).json())

prod -> 0


{'reply': "I don't have access to real-time shipping or port data to confirm HF-26012's arrival in Felixstowe.", 'guarantees_date': False}


## Step 6. Release check, then clean up

In [7]:
ASKS = ["Will HF-26012 arrive in Felixstowe by Friday?",
        "Can you promise HF-26025 is released before the weekend?",
        "Our sales team needs a firm date for HF-26003. What date can you commit to?"]

def release_check(version):
    mistral.beta.agents.create_version_alias(agent_id=agent.id, alias="prod", version=version)
    bad = sum(client.post("/reply", json={"text": t}).json()["guarantees_date"] for t in ASKS)
    return f"version {version}: {bad}/{len(ASKS)} replies guarantee a date"

print(release_check(v2.version))
print(release_check(agent.version))
mistral.beta.agents.delete(agent_id=agent.id)
print("routes:", [r.path for r in api.routes if r.path.startswith("/") and "docs" not in r.path
                  and "openapi" not in r.path])

version 1: 3/3 replies guarantee a date


version 0: 0/3 replies guarantee a date


routes: ['/redoc', '/triage', '/triage/v2', '/answer', '/notify', '/reply']
